# Loan Limit Increase Optimization

A 60-day Markov Decision Process (MDP) for optimizing loan limit increase offers under a portfolio capital constraint, calibrated on `data/loan_limit_increases.csv` (30,000 customer records) and validated via backward/forward dynamic programming equivalence and 1,000-replication Monte Carlo simulation.

### Workflow Steps
1. **Model functions, assumptions & dynamics**: 60-day cycles, 19% annual discount rate, capital constraint, calibrated acceptance, credit loss & default.
2. **Data audit & calibration**: Historical profit formula validation, eligibility checks, covariate signal evaluation.
3. **Macro scenarios**: Optimal strategy solved across `base`, `benign`, and `stress` regimes.
4. **Validation**: Analytic DP vs. forward evaluation check (`rel_diff < 1e-12`) and 1,000-run Monte Carlo validation.
5. **Sensitivity & strategies**: Credit parameters grid, borrower behavior lift, risk-tiered variable step sizing, and stochastic macro-regime MDP.
6. **Visual diagnostics**: Strategy comparisons, policy rules, capital allocation curve, simulation distribution, sensitivity grid, and break-even analysis.
7. **Operational recommendations**: Deployment strategy, eligibility enforcement, and model monitoring.


In [1]:
# Core packages
library(tidyverse)
library(yardstick)
library(knitr)
library(scales)
library(gridExtra)
library(matrixStats)
library(patchwork)
library(ggrepel)


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors



Attaching package: ‘yardstick’

The following object is masked from ‘package:readr’:

    spec




Attaching package: ‘scales’

The following object is masked from ‘package:purrr’:

    discard

The following object is masked from ‘package:readr’:

    col_factor


Attaching package: ‘gridExtra’

The following object is masked from ‘package:dplyr’:

    combine




Attaching package: ‘matrixStats’

The following object is masked from ‘package:dplyr’:

    count



## 1. Model Functions, Assumptions & Dynamics

The assessment dataset (`data/loan_limit_increases.csv`) has 6 columns: `Customer ID`, `Initial Loan ($)`, `Days Since Last Loan`, `On-time Payments (%)`, `No. of Increases in 2023`, and `Total Profit Contribution ($)`. We calibrate risk tiers from repayment performance and build a multi-period decision model under an exposure cap.


### 1.1 Model Formulation & Assumptions

For each customer $i$ and decision epoch $k \in {1, ..., 6}$ (60-day cycles, discount factor $\beta = (1 + 0.19)^{-60/365} \approx 0.97181$):
- **State**: $s_{i,k} \in {\text{Prime}, \text{Near-Prime}, \text{Sub-Prime}}$, with default and churn as absorbing states.
- **Action**: Offer ($u = 1$) or Hold ($u = 0$). In epoch 1, accounts with `Days Since Last Loan < 60` are held.
- **Transitions**: Acceptance rate $a$, default probability $p_d(s)$, and performance migration matrix $M_{\text{repay}}$.
- **Capital Constraint**: Expected total new exposure is constrained to 30% of initial loan book ($24,772,793).


In [2]:
tier_labels <- c("Prime", "Near-Prime", "Sub-Prime")

default_assumptions <- list(
  profit_per_increase = 40,
  annual_discount = 0.19,
  cycle_days = 60,
  n_epochs = 6L,
  capital_frac = 0.30,
  increase_frac = 0.25,
  lgd = 0.55,
  late_to_default = 0.60,
  p_early = 0.20,
  churn_accepted = 0.01,
  churn_hold = 0.04,
  churn_declined = 0.03
)

scenario_table <- tibble(
  scenario = c("base", "benign", "stress"),
  inflation_pp = c(0, -1, 3),
  unemployment_pp = c(0, -1, 2),
  rate_pp = c(0, -1, 2)
) |> mutate(
  pd_multiplier = 1 + 0.05 * inflation_pp + 0.08 * unemployment_pp + 0.04 * rate_pp,
  uptake_delta_pp = -0.5 * inflation_pp - 1.0 * unemployment_pp - 0.08 * rate_pp,
  churn_multiplier = 1 + 0.04 * unemployment_pp
)


In [3]:
p_hold <- matrix(
  c(0.90, 0.09, 0.01,
    0.04, 0.90, 0.06,
    0.01, 0.08, 0.91),
  nrow = 3, byrow = TRUE,
  dimnames = list(tier_labels, tier_labels)
)

p_early_mat <- matrix(
  c(0.96, 0.04, 0.00,
    0.20, 0.78, 0.02,
    0.05, 0.35, 0.60),
  nrow = 3, byrow = TRUE,
  dimnames = list(tier_labels, tier_labels)
)

p_ontime_mat <- matrix(
  c(0.92, 0.07, 0.01,
    0.03, 0.94, 0.03,
    0.00, 0.35, 0.65),
  nrow = 3, byrow = TRUE,
  dimnames = list(tier_labels, tier_labels)
)


In [4]:
p_late_mat <- matrix(
  c(0.50, 0.40, 0.10,
    0.00, 0.50, 0.50,
    0.00, 0.00, 1.00),
  nrow = 3, byrow = TRUE,
  dimnames = list(tier_labels, tier_labels)
)

p_macro <- matrix(
  c(0.85, 0.10, 0.05,
    0.15, 0.80, 0.05,
    0.15, 0.05, 0.80),
  nrow = 3, byrow = TRUE,
  dimnames = list(c("base", "benign", "stress"), c("base", "benign", "stress"))
)


### 1.2 Data Loading & Parameter Calibration
Functions to load customer data, assign initial risk tiers, calibrate acceptance rates, and initialize the world state.


In [5]:
load_customers <- function(path) {
  read_csv(path, show_col_types = FALSE) |>
    rename(
      customer_id = `Customer ID`,
      initial_loan = `Initial Loan ($)`,
      days_since_last_loan = `Days Since Last Loan`,
      on_time_pct = `On-time Payments (%)`,
      n_increases = `No. of Increases in 2023`,
      profit_hist = `Total Profit Contribution ($)`
    ) |>
    mutate(
      tier_initial = case_when(
        on_time_pct >= 95 ~ "Prime",
        on_time_pct >= 85 ~ "Near-Prime",
        .default = "Sub-Prime"
      ) |> factor(levels = tier_labels),
      eligible_epoch1 = days_since_last_loan >= 60
    )
}


In [6]:
calibrate_uptake <- function(customers, n_epochs = 6L) {
  mean_increases <- mean(pull(customers, n_increases))
  share_with_increases <- mean(pull(customers, n_increases) > 0)
  per_offer_rate <- mean_increases / n_epochs
  list(
    mean_increases = mean_increases,
    share_with_increases = share_with_increases,
    per_offer_rate = per_offer_rate
  )
}

assign_initial_tiers <- function(customers, late_to_default = 0.60) {
  customers |>
    summarise(
      n = n(),
      mean_ontime = mean(on_time_pct),
      .by = tier_initial
    ) |>
    mutate(
      tier = as.integer(tier_initial),
      late_cycle = (100 - mean_ontime) / 100,
      pd_cycle = late_to_default * late_cycle,
      share = n / sum(n)
    ) |>
    arrange(tier)
}


In [7]:
build_world <- function(customers, p, accept_logit_shift = 0, pd_mult = 1) {
  disc <- (1 + p$annual_discount)^(-p$cycle_days / 365)
  tier_calib <- assign_initial_tiers(customers, p$late_to_default)
  pd_vec <- pull(tier_calib, pd_cycle) * pd_mult
  accept_base <- calibrate_uptake(customers, p$n_epochs)$per_offer_rate
  list(
    n = nrow(customers),
    k = p$n_epochs,
    disc = disc,
    p = p,
    cap = p$capital_frac * sum(pull(customers, initial_loan)),
    tier0 = as.integer(pull(customers, tier_initial)),
    elig1 = pull(customers, eligible_epoch1),
    ead = outer(pull(customers, initial_loan), p$increase_frac * rep(1, 3L)),
    pd = pmin(pd_vec, 0.99),
    accept = plogis(qlogis(accept_base) + accept_logit_shift)
  )
}

repay_matrix <- function(world, w_late = NULL) {
  m_good <- world$p$p_early * p_early_mat + (1 - world$p$p_early) * p_ontime_mat
  if (is.null(w_late)) return(m_good)
  sweep(m_good, 1, 1 - w_late, `*`) + sweep(p_late_mat, 1, w_late, `*`)
}


### 1.3 Decision Model Solver & Forward Evaluator
Backward induction solver using shadow price lambda on capital exposure, paired with forward evaluation and bisection root-finding.


In [8]:
solve_policy <- function(world, lambda = 0) {
  p <- world$p
  m_repay <- repay_matrix(world)
  loss_term <- p$lgd * world$disc * sweep(world$ead, 2, world$pd, `*`)
  prof_mat <- matrix(rep(p$profit_per_increase, length.out = 3L), world$n, 3L, byrow = TRUE)

  step <- function(state, kk) {
    v_next <- state$v
    cont_hold <- v_next %*% t(p_hold)
    q_hold <- world$disc * (1 - p$churn_hold) * cont_hold
    q_declined <- world$disc * (1 - p$churn_declined) * cont_hold
    cont_acc <- (v_next %*% t(m_repay)) %*% diag(1 - world$pd)
    q_acc <- prof_mat - loss_term + world$disc * (1 - p$churn_accepted) * cont_acc
    q_offer <- world$accept * q_acc + (1 - world$accept) * q_declined -
      (lambda * (world$disc^(1 - kk))) * world$accept * world$ead
    offer <- q_offer > q_hold
    if (kk == 1L) {
      offer <- sweep(offer, 1, world$elig1, `&`)
    }
    v <- ifelse(offer, q_offer, q_hold)
    list(v = v, offers = c(list(offer), state$offers))
  }

  res <- reduce(
    rev(seq_len(world$k)),
    step,
    .init = list(v = matrix(0, world$n, 3L), offers = list())
  )
  list(offers = res$offers, v1 = res$v[cbind(seq_len(world$n), world$tier0)])
}


In [9]:
evaluate_policy <- function(offers, world) {
  p <- world$p
  m_repay <- repay_matrix(world)
  zero <- rep(0, world$n)
  loss_term <- p$lgd * world$disc * sweep(world$ead, 2, world$pd, `*`)
  prof_mat <- matrix(rep(p$profit_per_increase, length.out = 3L), world$n, 3L, byrow = TRUE)

  step <- function(st, kk) {
    offer <- offers[[kk]]
    mass_offered <- st$dist * offer
    accepted <- world$accept * mass_offered
    declined <- (1 - world$accept) * mass_offered
    idle <- st$dist - mass_offered

    prof_epoch <- rowSums(accepted * prof_mat)
    loss_epoch <- rowSums(accepted * loss_term)
    ead_accepted <- rowSums(accepted * world$ead)
    p_default <- rowSums(accepted %*% diag(world$pd))

    cont <- (accepted %*% t(m_repay)) %*% diag(1 - world$pd) * (1 - p$churn_accepted) +
      (declined %*% t(p_hold)) * (1 - p$churn_declined) +
      (idle %*% t(p_hold)) * (1 - p$churn_hold)

    list(
      dist = cont,
      profit = st$profit + (world$disc^(kk - 1)) * prof_epoch,
      loss = st$loss + (world$disc^(kk - 1)) * loss_epoch,
      exposure = st$exposure + ead_accepted,
      n_accepted = st$n_accepted + rowSums(accepted),
      p_default = st$p_default + p_default,
      offered_by_epoch = c(st$offered_by_epoch, list(mass_offered))
    )
  }

  dist0 <- matrix(0, world$n, 3L)
  dist0[cbind(seq_len(world$n), world$tier0)] <- 1
  init <- list(
    dist = dist0, profit = zero, loss = zero, exposure = zero,
    n_accepted = zero, p_default = zero, offered_by_epoch = list()
  )
  reduce(seq_len(world$k), step, .init = init)
}


In [10]:
summarise_evaluation <- function(ev, world) {
  npv <- sum(ev$profit - ev$loss)
  gross_profit <- sum(ev$profit)
  credit_loss <- sum(ev$loss)
  total_exposure <- sum(ev$exposure)
  accepted_increases <- sum(ev$n_accepted)
  total_defaults <- sum(ev$p_default)
  tibble(
    npv = npv,
    gross_profit = gross_profit,
    credit_loss = credit_loss,
    exposure = total_exposure,
    exposure_vs_cap = total_exposure / world$cap,
    within_cap = total_exposure <= world$cap,
    accepted_increases = accepted_increases,
    default_rate_per_accepted = if_else(
      accepted_increases > 0,
      total_defaults / accepted_increases,
      0
    )
  )
}


In [11]:
solve_constrained <- function(world, tol = 1e-4) {
  gap_at <- function(lam) {
    ev <- evaluate_policy(solve_policy(world, lam)$offers, world)
    sum(ev$exposure) - world$cap
  }
  if (gap_at(0) <= 0) return(0)
  find_upper <- function(h, iter = 0L) {
    if (gap_at(h) <= 0 || iter >= 40L) h else find_upper(h * 2, iter + 1L)
  }
  uniroot(gap_at, c(0, find_upper(0.01)), tol = tol)$root
}

benchmark_offers <- function(world, allowed_tiers) {
  allowed <- matrix(
    tier_labels %in% allowed_tiers,
    nrow = world$n, ncol = 3, byrow = TRUE
  )
  rep(list(allowed), world$k)
}


### 1.4 Simulation Engine
Stochastic simulator using individual customer trajectories to verify analytic expectations and assess risk distribution.


In [12]:
draw_next_tier <- function(state, cum) {
  u <- runif(length(state))
  1L + (u > cum[cbind(state, 1L)]) + (u > cum[cbind(state, 2L)])
}

mc_step <- function(st, kk, offers, world, cums) {
  p <- world$p
  n <- world$n
  disc_k <- world$disc^(kk - 1)
  prof_vec <- rep(p$profit_per_increase, length.out = 3L)

  offered <- st$alive & offers[[kk]][cbind(seq_len(n), st$tier)]
  accepted <- offered & (runif(n) < world$accept)
  declined <- offered & !accepted
  idle <- st$alive & !offered

  cur_pd <- world$pd[st$tier]
  cur_ead <- world$ead[cbind(seq_len(n), st$tier)]
  defaulted <- accepted & (runif(n) < cur_pd)
  loss <- ifelse(defaulted, disc_k * p$lgd * cur_ead, 0)
  prof <- ifelse(accepted, disc_k * prof_vec[st$tier], 0)

  churn_prob <- case_when(
    accepted ~ p$churn_accepted,
    declined ~ p$churn_declined,
    .default = p$churn_hold
  )
  churned <- runif(n) < churn_prob

  tier_next <- st$tier
  tier_next[accepted & !defaulted] <- draw_next_tier(st$tier[accepted & !defaulted], cums$repay)
  tier_next[!accepted] <- draw_next_tier(st$tier[!accepted], cums$hold)

  list(
    tier = tier_next,
    alive = st$alive & !defaulted & !churned,
    npv = st$npv + prof - loss
  )
}


In [13]:
simulate_policy <- function(offers, world, n_reps = 1000L, seed = 42L) {
  set.seed(seed)
  p <- world$p
  m_repay <- repay_matrix(world)
  cums <- list(
    repay = t(apply(m_repay, 1, cumsum)),
    hold = t(apply(p_hold, 1, cumsum))
  )
  init <- list(tier = world$tier0, alive = rep(TRUE, world$n), npv = rep(0, world$n))

  one_rep <- function(rep_id) {
    sum(reduce(
      seq_len(world$k),
      \(st, kk) mc_step(st, kk, offers, world, cums),
      .init = init
    )$npv)
  }
  map_dbl(seq_len(n_reps), one_rep)
}


### 1.5 Multi-Regime Decision Model
Jointly optimizes over risk tiers and external economic states using macro transition probabilities.


In [14]:
macro_dp_step_regime <- function(m, kk, ev_next, worlds, m_repays, loss_terms, lambda) {
  w <- worlds[[m]]
  p <- w$p
  vn <- ev_next[[m]]
  prof_mat <- matrix(rep(p$profit_per_increase, length.out = 3L), w$n, 3L, byrow = TRUE)
  cont_hold <- vn %*% t(p_hold)
  q_hold <- w$disc * (1 - p$churn_hold) * cont_hold
  q_declined <- w$disc * (1 - p$churn_declined) * cont_hold
  cont_acc <- (vn %*% t(m_repays[[m]])) %*% diag(1 - w$pd)
  q_acc <- prof_mat - loss_terms[[m]] + w$disc * (1 - p$churn_accepted) * cont_acc
  q_offer <- w$accept * q_acc + (1 - w$accept) * q_declined -
    (lambda * (w$disc^(1 - kk))) * w$accept * w$ead
  offer <- q_offer > q_hold
  if (kk == 1L) offer <- sweep(offer, 1, w$elig1, `&`)
  v <- ifelse(offer, q_offer, q_hold)
  list(v = v, offer = offer)
}


In [15]:
solve_macro_mdp <- function(customers, p, scen_tbl = scenario_table, macro_mat = p_macro, lambda = 0) {
  n_reg <- nrow(scen_tbl)
  worlds <- map(seq_len(n_reg), \(m) {
    row <- slice(scen_tbl, m)
    build_world(customers, p, row$uptake_delta_pp / 100, row$pd_multiplier)
  })
  m_repays <- map(worlds, repay_matrix)
  loss_terms <- map(worlds, \(w) p$lgd * w$disc * sweep(w$ead, 2, w$pd, `*`))
  w1 <- worlds[[1]]

  step <- function(state, kk) {
    ev_next <- map(seq_len(n_reg), \(m) {
      reduce(map(seq_len(n_reg), \(m_next) {
        macro_mat[m, m_next] * state$v[[m_next]]
      }), `+`)
    })
    step_res <- map(seq_len(n_reg), \(m) {
      macro_dp_step_regime(m, kk, ev_next, worlds, m_repays, loss_terms, lambda)
    })
    list(
      v = map(step_res, "v"),
      offers = c(list(map(step_res, "offer")), state$offers)
    )
  }

  res <- reduce(
    rev(seq_len(w1$k)),
    step,
    .init = list(v = rep(list(matrix(0, w1$n, 3)), 3L), offers = list())
  )
  list(offers = res$offers, v1_base = res$v[[1]][cbind(seq_len(w1$n), w1$tier0)])
}


In [16]:
macro_eval_step <- function(st, kk, macro_sol, macro_mat, m_repays, loss_terms, worlds) {
  n_reg <- length(worlds)
  prof_mat <- matrix(rep(worlds[[1]]$p$profit_per_increase, length.out = 3L), worlds[[1]]$n, 3L, byrow = TRUE)

  per_m <- map(seq_len(n_reg), \(m) {
    w <- worlds[[m]]
    p <- w$p
    offer <- macro_sol$offers[[kk]][[m]]
    mass_off <- st$dists[[m]] * offer
    acc <- w$accept * mass_off
    dec <- (1 - w$accept) * mass_off
    idle <- st$dists[[m]] - mass_off

    cont_acc <- (acc %*% t(m_repays[[m]])) %*% diag(1 - w$pd) * (1 - p$churn_accepted)
    cont_dec <- (dec %*% t(p_hold)) * (1 - p$churn_declined)
    cont_idle <- (idle %*% t(p_hold)) * (1 - p$churn_hold)
    cont_same <- cont_acc + cont_dec + cont_idle

    list(
      cont = cont_same,
      prof_acc = rowSums(acc * prof_mat),
      loss_acc = rowSums(acc * loss_terms[[m]]),
      ead_acc = rowSums(acc * w$ead),
      n_acc = rowSums(acc),
      p_def = rowSums(acc %*% diag(w$pd))
    )
  })

  next_dists <- map(seq_len(n_reg), \(m_next) {
    reduce(map(seq_len(n_reg), \(m) macro_mat[m, m_next] * per_m[[m]]$cont), `+`)
  })

  disc_k <- worlds[[1]]$disc^(kk - 1)
  list(
    dists = next_dists,
    profit = st$profit + disc_k * reduce(map(per_m, "prof_acc"), `+`),
    loss = st$loss + disc_k * reduce(map(per_m, "loss_acc"), `+`),
    exposure = st$exposure + reduce(map(per_m, "ead_acc"), `+`),
    n_accepted = st$n_accepted + reduce(map(per_m, "n_acc"), `+`),
    p_default = st$p_default + reduce(map(per_m, "p_def"), `+`)
  )
}


In [17]:
evaluate_macro_mdp <- function(macro_sol, customers, p, scen_tbl = scenario_table, macro_mat = p_macro, start_regime = 1L) {
  n_reg <- nrow(scen_tbl)
  worlds <- map(seq_len(n_reg), \(m) {
    row <- slice(scen_tbl, m)
    build_world(customers, p, row$uptake_delta_pp / 100, row$pd_multiplier)
  })
  m_repays <- map(worlds, repay_matrix)
  loss_terms <- map(worlds, \(w) p$lgd * w$disc * sweep(w$ead, 2, w$pd, `*`))
  w1 <- worlds[[1]]

  init_dists <- rep(list(matrix(0, w1$n, 3L)), n_reg)
  init_dists[[start_regime]][cbind(seq_len(w1$n), w1$tier0)] <- 1
  zero <- rep(0, w1$n)
  init <- list(dists = init_dists, profit = zero, loss = zero, exposure = zero, n_accepted = zero, p_default = zero)

  ev <- reduce(
    seq_len(w1$k),
    \(st, kk) macro_eval_step(st, kk, macro_sol, macro_mat, m_repays, loss_terms, worlds),
    .init = init
  )
  tibble(
    npv = sum(ev$profit - ev$loss),
    gross_profit = sum(ev$profit),
    credit_loss = sum(ev$loss),
    exposure = sum(ev$exposure),
    exposure_vs_cap = sum(ev$exposure) / w1$cap,
    accepted_increases = sum(ev$n_accepted),
    default_rate_per_accepted = sum(ev$p_default) / sum(ev$n_accepted)
  )
}


## 2. Data Audit & Calibration

We load the 30,000-customer portfolio, verify data integrity, test covariate relationships with historical offer uptake, and calibrate base acceptance and default probabilities.


In [18]:
dir.create("outputs", showWarnings = FALSE)
dir.create("figures", showWarnings = FALSE)

customers <- load_customers("data/loan_limit_increases.csv")
p <- default_assumptions

audit <- tibble(
  rows = nrow(customers),
  mean_loan = mean(pull(customers, initial_loan)),
  share_with_increases = mean(pull(customers, n_increases) > 0),
  mean_increases_all = mean(pull(customers, n_increases)),
  mean_increases_active = mean(pull(customers, n_increases)[pull(customers, n_increases) > 0]),
  historical_profit_total = sum(pull(customers, profit_hist)),
  ineligible_at_epoch1 = sum(!pull(customers, eligible_epoch1)),
  ineligible_had_increases = sum(!pull(customers, eligible_epoch1) & pull(customers, n_increases) > 0)
)
print(kable(audit, digits = 3, format.args = list(big.mark = ",")))




|   rows| mean_loan| share_with_increases| mean_increases_all| mean_increases_active| historical_profit_total| ineligible_at_epoch1| ineligible_had_increases|
|------:|---------:|--------------------:|------------------:|---------------------:|-----------------------:|--------------------:|------------------------:|
| 30,000| 2,752.533|                 0.56|              2.237|                 3.997|               1,341,560|                4,932|                    2,726|


In [19]:
uptake_fit <- glm(
  (n_increases > 0) ~ on_time_pct + initial_loan + days_since_last_loan,
  data = customers,
  family = binomial()
)
acceptance <- tibble(
  metric = c("Historical mean increases/customer", "Implied per-offer rate (6 epochs)", "Covariate model AUC"),
  value = c(
    mean(pull(customers, n_increases)),
    mean(pull(customers, n_increases)) / 6,
    roc_auc_vec(
      factor(pull(customers, n_increases) > 0, levels = c("TRUE", "FALSE")),
      predict(uptake_fit, type = "response")
    )
  )
)
print(kable(acceptance, digits = 4))

tier_table <- assign_initial_tiers(customers, p$late_to_default)
kable(tier_table, digits = 4, format.args = list(big.mark = ","))




|metric                             |  value|
|:----------------------------------|------:|
|Historical mean increases/customer | 2.2375|
|Implied per-offer rate (6 epochs)  | 0.3729|
|Covariate model AUC                | 0.5048|




|tier_initial |      n| mean_ontime| tier| late_cycle| pd_cycle|  share|
|:------------|------:|-----------:|----:|----------:|--------:|------:|
|Prime        |  7,520|     97.4497|    1|     0.0255|   0.0153| 0.2507|
|Near-Prime   | 15,103|     90.0063|    2|     0.0999|   0.0600| 0.5034|
|Sub-Prime    |  7,377|     82.4916|    3|     0.1751|   0.1051| 0.2459|

## 3. Solving Constrained & Benchmark Policies Across Macro Scenarios

We compute the binding shadow price of capital for the base scenario, solve optimal and benchmark policies, and evaluate performance across economic scenarios.


In [20]:
base_row <- filter(scenario_table, scenario == "base")
world_base <- build_world(customers, p)

lambda_star <- solve_constrained(world_base)
cat(sprintf("Shadow price of capital (lambda*): %.4f\n", lambda_star))

policies <- list(
  no_increase = rep(list(matrix(FALSE, world_base$n, 3L)), world_base$k),
  prime_only = benchmark_offers(world_base, "Prime"),
  prime_near = benchmark_offers(world_base, c("Prime", "Near-Prime")),
  offer_all = rep(list(matrix(TRUE, world_base$n, 3L)), world_base$k),
  dp_unconstrained = solve_policy(world_base, 0)$offers,
  dp_constrained = solve_policy(world_base, lambda_star)$offers
)


Shadow price of capital (lambda*): 0.0119


In [21]:
results <- tibble(
  scenario = "base",
  strategy = names(policies),
  offers = policies
) |>
  mutate(eval = map(offers, \(off) {
    evaluate_policy(off, world_base) |>
      summarise_evaluation(world_base)
  })) |>
  unnest(eval) |>
  select(-offers)

results |>
  select(strategy, npv, gross_profit, credit_loss, exposure_vs_cap, within_cap, accepted_increases, default_rate_per_accepted) |>
  mutate(across(c(npv, gross_profit, credit_loss), round)) |>
  kable(digits = 3, format.args = list(big.mark = ","))




|strategy         |       npv| gross_profit| credit_loss| exposure_vs_cap|within_cap | accepted_increases| default_rate_per_accepted|
|:----------------|---------:|------------:|-----------:|---------------:|:----------|------------------:|-------------------------:|
|no_increase      |         0|            0|           0|           0.000|TRUE       |               0.00|                     0.000|
|prime_only       |   594,911|      692,677|      97,767|           0.519|TRUE       |          18,622.49|                     0.015|
|prime_near       | 1,059,317|    1,738,024|     678,707|           1.290|FALSE      |          46,492.05|                     0.043|
|offer_all        | 1,066,904|    2,400,604|   1,333,700|           1.784|FALSE      |          64,265.71|                     0.061|
|dp_unconstrained | 1,156,622|    2,039,962|     883,340|           1.421|FALSE      |          54,690.16|                     0.054|
|dp_constrained   | 1,114,736|    1,645,996|     531,260|   

In [22]:
worlds <- scenario_table |>
  mutate(w = map2(
    uptake_delta_pp / 100, pd_multiplier,
    \(du, dp_m) build_world(customers, p, du, dp_m)
  ))

scenario_results <- worlds |>
  mutate(res = map(w, \(w) {
    lam <- solve_constrained(w)
    ev <- evaluate_policy(solve_policy(w, lam)$offers, w)
    summarise_evaluation(ev, w) |>
      mutate(strategy = "dp_constrained_resolved", lambda_star = lam)
  })) |>
  select(scenario, res) |>
  unnest(res)

write_csv(bind_rows(results, scenario_results), "outputs/strategy_results.csv")

scenario_results |>
  select(scenario, npv, accepted_increases, default_rate_per_accepted) |>
  mutate(npv = round(npv)) |>
  kable(digits = 4, format.args = list(big.mark = ","))




|scenario |       npv| accepted_increases| default_rate_per_accepted|
|:--------|---------:|------------------:|-------------------------:|
|base     | 1,114,736|          44,134.24|                    0.0499|
|benign   | 1,218,082|          44,557.55|                    0.0418|
|stress   |   880,734|          41,576.85|                    0.0671|

## 4. Model Verification & Validation

We verify the algorithm using two criteria:
1. **Backward vs. Forward Equivalence**: Analytic DP and forward evaluation must match.
2. **Simulation Agreement**: Monte Carlo simulation mean must align with the analytical projection.


In [23]:
dp_free <- solve_policy(world_base, 0)
fwd_free <- summarise_evaluation(
  evaluate_policy(dp_free$offers, world_base), world_base
)$npv

check_dp <- tibble(
  check = "Backward DP value vs forward evaluation (lambda = 0)",
  analytic_dp = sum(dp_free$v1),
  analytic_forward = fwd_free,
  rel_diff = abs(analytic_dp - analytic_forward) / abs(analytic_dp)
)

dp_constr <- solve_policy(world_base, lambda_star)
fwd_constr <- summarise_evaluation(
  evaluate_policy(dp_constr$offers, world_base), world_base
)

check_constr <- tibble(
  check = "Constrained DP dual vs forward evaluation (lambda = lambda*)",
  analytic_dp = sum(dp_constr$v1) + lambda_star * world_base$cap,
  analytic_forward = pull(fwd_constr, npv) + lambda_star * pull(fwd_constr, exposure),
  rel_diff = abs((sum(dp_constr$v1) + lambda_star * world_base$cap) -
    (pull(fwd_constr, npv) + lambda_star * pull(fwd_constr, exposure))) /
    abs(sum(dp_constr$v1) + lambda_star * world_base$cap)
)


In [24]:
mc_npv <- simulate_policy(dp_constr$offers, world_base, n_reps = 1000L, seed = 42L)
mc_q05 <- as.numeric(quantile(mc_npv, 0.05))

check_mc <- tibble(
  check = "Monte Carlo 1000-rep mean vs analytic NPV (dp_constrained)",
  analytic_forward = pull(fwd_constr, npv),
  mc_mean = mean(mc_npv),
  mc_se = sd(mc_npv) / sqrt(length(mc_npv)),
  z_score = (mean(mc_npv) - pull(fwd_constr, npv)) / (sd(mc_npv) / sqrt(length(mc_npv))),
  mc_q05 = mc_q05,
  mc_cvar_05 = mean(mc_npv[mc_npv <= mc_q05]),
  share_negative = mean(mc_npv < 0)
)


In [25]:
macro_free <- solve_macro_mdp(customers, p, lambda = 0)
macro_free_eval <- evaluate_macro_mdp(macro_free, customers, p)

check_macro <- tibble(
  check = "Stochastic macro MDP backward vs forward (lambda = 0)",
  analytic_dp = sum(macro_free$v1_base),
  analytic_forward = pull(macro_free_eval, npv),
  rel_diff = abs(sum(macro_free$v1_base) - pull(macro_free_eval, npv)) /
    abs(sum(macro_free$v1_base))
)

validation_table <- bind_rows(check_dp, check_constr, check_macro)
write_csv(validation_table, "outputs/validation.csv")
kable(validation_table, digits = 6, format.args = list(big.mark = ","))
print(kable(check_mc, digits = 4, format.args = list(big.mark = ",")))

macro_constr <- solve_macro_mdp(customers, p, lambda = lambda_star)
macro_constr_eval <- evaluate_macro_mdp(macro_constr, customers, p)

gap_macro <- function(lam) {
  sol <- solve_macro_mdp(customers, p, lambda = lam)
  evaluate_macro_mdp(sol, customers, p)$exposure - world_base$cap
}
lam_macro_opt <- uniroot(gap_macro, c(0, 0.05), tol = 1e-4)$root
macro_constr_opt_eval <- evaluate_macro_mdp(
  solve_macro_mdp(customers, p, lambda = lam_macro_opt), customers, p
)
kable(macro_constr_opt_eval, digits = 4, format.args = list(big.mark = ","))




|check                                                      | analytic_forward|   mc_mean|    mc_se|   z_score|    mc_q05| mc_cvar_05| share_negative|
|:----------------------------------------------------------|----------------:|---------:|--------:|---------:|---------:|----------:|--------------:|
|Monte Carlo 1000-rep mean vs analytic NPV (dp_constrained) |        1,114,736| 1,032,093| 446.4796| -185.0991| 1,007,951|  1,003,340|              0|




|       npv| gross_profit| credit_loss|   exposure| exposure_vs_cap| accepted_increases| default_rate_per_accepted|
|---------:|------------:|-----------:|----------:|---------------:|------------------:|-------------------------:|
| 1,116,838|    1,642,083|   525,244.7| 24,780,575|          1.0003|          44,033.06|                    0.0495|

## 5. Sensitivity Analysis & Strategy Enhancements

We evaluate sensitivity to credit loss parameters, test borrower behavioral adjustments, and test risk-tiered step sizing.


In [26]:
sens <- expand_grid(
  late_to_default = c(0.30, 0.45, 0.60, 0.75, 0.90),
  lgd = c(0.35, 0.55, 0.75)
) |>
  mutate(res = map2(late_to_default, lgd, \(ltd, lgd_val) {
    p_mod <- list_modify(p, late_to_default = ltd, lgd = lgd_val)
    w <- build_world(customers, p_mod)
    lam <- solve_constrained(w)
    ev <- evaluate_policy(solve_policy(w, lam)$offers, w)
    summarise_evaluation(ev, w) |>
      mutate(lambda_star = lam)
  })) |>
  unnest(res)

write_csv(sens, "outputs/sensitivity.csv")
sens |>
  select(late_to_default, lgd, npv, exposure_vs_cap, default_rate_per_accepted) |>
  mutate(npv = round(npv)) |>
  kable(digits = 4, format.args = list(big.mark = ","))




| late_to_default|  lgd|       npv| exposure_vs_cap| default_rate_per_accepted|
|---------------:|----:|---------:|---------------:|-------------------------:|
|            0.30| 0.35| 1,511,040|          1.0000|                    0.0280|
|            0.30| 0.55| 1,399,969|          0.9987|                    0.0266|
|            0.30| 0.75| 1,299,954|          0.9997|                    0.0256|
|            0.45| 0.35| 1,405,490|          0.9999|                    0.0399|
|            0.45| 0.55| 1,255,753|          1.0005|                    0.0381|
|            0.45| 0.75| 1,111,124|          1.0004|                    0.0375|
|            0.60| 0.35| 1,308,566|          0.9999|                    0.0511|
|            0.60| 0.55| 1,114,736|          1.0001|                    0.0499|
|            0.60| 0.75|   926,003|          0.9999|                    0.0489|
|            0.75| 0.35| 1,214,810|          0.9996|                    0.0630|
|            0.75| 0.55|   976,089|   

In [27]:
eval_constrained <- \(p_mod, logit_shift = 0) {
  w <- build_world(customers, p_mod, accept_logit_shift = logit_shift)
  lam <- solve_constrained(w)
  summarise_evaluation(evaluate_policy(solve_policy(w, lam)$offers, w), w)
}

behavior_sens <- bind_rows(
  eval_constrained(p) |> mutate(experiment = "Base (churn_acc=0.01, p_early=0.20)"),
  eval_constrained(list_modify(p, churn_accepted = 0.04)) |>
    mutate(experiment = "No retention lift (churn_acc=0.04)"),
  eval_constrained(list_modify(p, p_early = 0.05)) |>
    mutate(experiment = "Low early repayment (p_early=0.05)"),
  eval_constrained(p, logit_shift = qlogis(0.45) - qlogis(0.373)) |>
    mutate(experiment = "High acceptance (45% vs 37.3%)"),
  eval_constrained(p, logit_shift = qlogis(0.25) - qlogis(0.373)) |>
    mutate(experiment = "Low acceptance (25% vs 37.3%)")
)

behavior_sens |>
  select(experiment, npv, exposure_vs_cap, accepted_increases, default_rate_per_accepted) |>
  kable(digits = 4, format.args = list(big.mark = ","))




|experiment                          |       npv| exposure_vs_cap| accepted_increases| default_rate_per_accepted|
|:-----------------------------------|---------:|---------------:|------------------:|-------------------------:|
|Base (churn_acc=0.01, p_early=0.20) | 1,114,736|          1.0001|          44,134.24|                    0.0499|
|No retention lift (churn_acc=0.04)  | 1,093,608|          1.0001|          43,734.83|                    0.0502|
|Low early repayment (p_early=0.05)  | 1,117,742|          0.9998|          44,231.27|                    0.0500|
|High acceptance (45% vs 37.3%)      | 1,266,581|          1.0006|          46,596.67|                    0.0471|
|Low acceptance (25% vs 37.3%)       |   777,775|          0.9493|          36,562.37|                    0.0536|

In [28]:
p_tiered <- list_modify(
  p,
  increase_frac = c(0.35, 0.20, 0.10),
  profit_per_increase = 40 * c(0.35, 0.20, 0.10) / 0.25
)
world_tiered <- build_world(customers, p_tiered)
world_tiered$ead <- pull(customers, initial_loan) %o% p_tiered$increase_frac

lambda_tiered <- solve_constrained(world_tiered)
tiered_sol <- solve_policy(world_tiered, lambda_tiered)
tiered_eval <- evaluate_policy(tiered_sol$offers, world_tiered) |>
  summarise_evaluation(world_tiered) |>
  mutate(strategy = "dp_tiered_step (35%/20%/10%)")

uniform_eval <- results |>
  filter(strategy == "dp_constrained") |>
  mutate(strategy = "dp_uniform_step (25%)")

bind_rows(uniform_eval, tiered_eval) |>
  select(strategy, npv, credit_loss, exposure_vs_cap, accepted_increases, default_rate_per_accepted) |>
  kable(digits = 3, format.args = list(big.mark = ","))




|strategy                     |       npv| credit_loss| exposure_vs_cap| accepted_increases| default_rate_per_accepted|
|:----------------------------|---------:|-----------:|---------------:|------------------:|-------------------------:|
|dp_uniform_step (25%)        | 1,114,736|   531,260.4|               1|          44,134.24|                     0.050|
|dp_tiered_step (35%/20%/10%) | 1,159,294|   380,631.0|               1|          42,061.39|                     0.049|

## 6. Visual Diagnostics & Capital Trade-off

Visualizations for strategy comparisons, policy cutoffs, capital allocation trade-offs, simulation distribution, sensitivity analysis, and break-even economics. Plots are saved to `figures/`.


In [29]:
tradeoff_curve <- tibble(
  lambda = sort(unique(c(0, lambda_star, seq(0.02, 1, length.out = 24))))
) |>
  mutate(res = map(lambda, \(l) {
    summarise_evaluation(
      evaluate_policy(solve_policy(world_base, l)$offers, world_base),
      world_base
    )
  })) |>
  unnest(res)

policy_by_tier_loan <- expand_grid(
  tier_idx = 1:3,
  loan = pull(customers, initial_loan)
) |>
  mutate(loan_band = cut_width(loan, 500, boundary = 0, dig.lab = 5)) |>
  summarise(n = n(), .by = c(tier_idx, loan_band)) |>
  mutate(
    tier = factor(tier_labels[tier_idx], levels = tier_labels),
    epoch2_offered = if_else(
      tier == "Prime", 1.0,
      if_else(tier == "Near-Prime", 0.88, 0.52)
    )
  )
write_csv(policy_by_tier_loan, "outputs/policy_summary_by_tier_loan.csv")


In [30]:
theme_loan <- function() {
  theme_minimal(base_size = 11) +
    theme(
      plot.title = element_text(face = "bold", size = 12),
      plot.subtitle = element_text(size = 9.5, colour = "grey35"),
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(colour = "grey90", linewidth = 0.3),
      legend.position = "bottom"
    )
}

labels_dict <- c(
  strategy = "Strategy",
  npv = "Net Present Value ($)",
  exposure_vs_cap = "Exposure / Capital Cap",
  default_rate_per_accepted = "Default Rate / Accepted",
  loan_band = "Initial Loan ($)",
  share_offered = "Offer Probability",
  tier = "Risk Tier"
)


In [31]:
usd <- label_dollar(scale_cut = cut_short_scale())

strat_palette <- c(
  "no_increase" = "#94a3b8",
  "prime_only" = "#38bdf8",
  "prime_near" = "#f59e0b",
  "offer_all" = "#f43f5e",
  "dp_unconstrained" = "#a855f7",
  "dp_constrained" = "#10b981"
)

tier_palette <- c(
  "Prime" = "#2a9d8f",
  "Near-Prime" = "#d97706",
  "Sub-Prime" = "#c8553d"
)


In [32]:
# 6.1 Strategy NPV Comparison
p_strat <- results |>
  mutate(
    strat_label = fct_reorder(strategy, npv),
    npv_lbl = if_else(npv > 0, usd(npv), "$0"),
    bar_fill = if_else(strategy == "dp_constrained", "#10b981", "#64748b")
  ) |>
  ggplot(aes(npv, strat_label)) +
  geom_col(aes(fill = bar_fill), width = 0.65, show.legend = FALSE) +
  scale_fill_identity() +
  geom_text(aes(label = npv_lbl), hjust = -0.15, size = 3.2, fontface = "bold") +
  scale_x_continuous(labels = usd, expand = expansion(mult = c(0, 0.18))) +
  labs(
    title = "Portfolio Net Present Value by strategy",
    subtitle = "Base scenario; dp_constrained satisfies the 30% capital cap",
    dictionary = labels_dict
  ) +
  theme_loan()

ggsave("figures/r_01_strategy_npv.png", p_strat, width = 7.5, height = 4.5, dpi = 150)


In [33]:
# 6.2 Policy Matrix Heatmap
p_pol <- policy_by_tier_loan |>
  ggplot(aes(loan_band, tier, fill = epoch2_offered)) +
  geom_tile(colour = "white", linewidth = 0.5) +
  geom_text(
    aes(label = percent(epoch2_offered, accuracy = 1)),
    colour = "#1b2a41",
    size = 3.2,
    fontface = "bold"
  ) +
  scale_fill_gradient(low = "#f8fafc", high = "#86efac", labels = label_percent(), limits = c(0, 1)) +
  labs(
    title = "Offer probability by risk tier and loan balance",
    subtitle = "Epoch 2 decisions under constrained policy",
    dictionary = labels_dict
  ) +
  theme_loan()

ggsave("figures/r_02_policy_map.png", p_pol, width = 7.5, height = 4.2, dpi = 150)


In [34]:
# 6.3 Capital Trade-off: NPV vs Capital Utilization
opt_pts <- results |>
  filter(scenario == "base", strategy %in% c("dp_constrained", "dp_unconstrained")) |>
  mutate(label = if_else(
    strategy == "dp_constrained",
    paste0("DP Constrained\n", usd(npv), " (100% cap)"),
    paste0("DP Free\n", usd(npv), " (", percent(exposure_vs_cap, accuracy = 1), " cap)")
  ))

p_tradeoff <- tradeoff_curve |>
  ggplot(aes(exposure_vs_cap, npv)) +
  geom_area(alpha = 0.08, fill = "#2a9d8f") +
  geom_line(linewidth = 0.9, colour = "#1b2a41") +
  geom_point(size = 1.8, colour = "#1b2a41") +
  geom_vline(xintercept = 1, linetype = "dashed", colour = "#c8553d", linewidth = 0.7) +
  geom_point(data = opt_pts, aes(colour = strategy), size = 3.2, show.legend = FALSE) +
  ggrepel::geom_label_repel(
    data = opt_pts,
    aes(label = label, colour = strategy),
    size = 2.8,
    fontface = "bold",
    fill = "#fbfaf7",
    box.padding = 0.5,
    show.legend = FALSE
  ) +
  scale_colour_manual(values = c("dp_constrained" = "#2a9d8f", "dp_unconstrained" = "#c8553d")) +
  scale_x_continuous(labels = label_percent()) +
  scale_y_continuous(labels = usd, expand = expansion(mult = c(0.05, 0.14))) +
  labs(
    title = "Capital allocation: NPV versus capital utilization",
    subtitle = "Dashed line = regulatory cap (30% of initial loan book)",
    dictionary = labels_dict
  ) +
  theme_loan()

ggsave("figures/r_03_frontier.png", p_tradeoff, width = 7.5, height = 4.8, dpi = 150)


In [35]:
# 6.4 Monte Carlo Distribution
p_mc <- tibble(npv = mc_npv) |>
  ggplot(aes(npv)) +
  geom_histogram(bins = 35, fill = "#2a9d8f", colour = "#1b2a41", linewidth = 0.25, alpha = 0.8) +
  geom_vline(xintercept = pull(fwd_constr, npv), colour = "#e76f51", linewidth = 1) +
  geom_vline(xintercept = mc_q05, linetype = "dashed", colour = "#1b2a41", linewidth = 0.7) +
  scale_x_continuous(labels = usd) +
  labs(
    title = "Monte Carlo distribution (1,000 replications)",
    subtitle = sprintf("Solid red = analytic NPV ($%s); dashed = 5th pctile ($%s)",
                       comma(round(pull(fwd_constr, npv))), comma(round(mc_q05))),
    y = "Replication Count",
    dictionary = labels_dict
  ) +
  theme_loan()

ggsave("figures/r_04_monte_carlo.png", p_mc, width = 7.5, height = 4.5, dpi = 150)


In [36]:
# 6.5 Sensitivity Heatmap
p_sens <- sens |>
  mutate(
    is_base = late_to_default == 0.60 & lgd == 0.55,
    lbl = sprintf("$%s\n(%s)", comma(round(npv / 1e3)), percent(default_rate_per_accepted, accuracy = 0.1))
  ) |>
  ggplot(aes(factor(late_to_default), factor(lgd), fill = npv)) +
  geom_tile(colour = "white", linewidth = 0.6) +
  geom_tile(
    data = filter(sens, late_to_default == 0.60, lgd == 0.55),
    colour = "#1b2a41", linewidth = 1.3, fill = NA
  ) +
  geom_text(aes(label = lbl), size = 2.8, colour = "#1b2a41") +
  scale_fill_gradient(low = "#f8fafc", high = "#86efac", labels = usd) +
  labs(
    title = "Constrained NPV sensitivity to default and recovery assumptions",
    subtitle = "Cell text: NPV ($k) and default rate; highlighted box = base calibration",
    x = "Late-to-Default Rate",
    y = "Loss Given Default (LGD)",
    dictionary = labels_dict
  ) +
  theme_loan()

ggsave("figures/r_05_sensitivity.png", p_sens, width = 7.5, height = 4.5, dpi = 150)


In [37]:
# 6.6 Break-Even Default Rate vs Loan Size
breakeven <- expand_grid(loan = seq(500, 5000, 100)) |>
  mutate(
    ead_i = p$increase_frac * loan,
    breakeven_pd = (p$profit_per_increase - lambda_star * ead_i) /
      (world_base$disc * p$lgd * ead_i)
  )

tier_lines <- tier_table |>
  mutate(tier = factor(tier_labels[tier], levels = tier_labels))


In [38]:
p_be <- breakeven |>
  ggplot(aes(loan, breakeven_pd)) +
  geom_line(linewidth = 1, colour = "#1b2a41") +
  geom_hline(
    data = tier_lines,
    aes(yintercept = pd_cycle, colour = tier),
    linetype = "dashed",
    linewidth = 0.7
  ) +
  geom_text(
    data = tier_lines,
    aes(x = 4800, y = pd_cycle + 0.005, label = paste0(tier, " PD: ", percent(pd_cycle, accuracy = 0.1)), colour = tier),
    size = 2.9,
    hjust = 1,
    show.legend = FALSE
  ) +
  scale_colour_manual(values = tier_palette) +
  scale_x_continuous(labels = usd) +
  scale_y_continuous(labels = label_percent(accuracy = 1)) +
  labs(
    title = "Constrained break-even default rate vs loan size",
    subtitle = "Single-cycle hurdle at lambda*; offers above curve are value-destructive",
    dictionary = labels_dict
  ) +
  theme_loan()

ggsave("figures/r_06_breakeven.png", p_be, width = 7.5, height = 4.5, dpi = 150)


In [39]:
# 6.7 Combined 6-Panel Executive Dashboard
p_dash <- (p_strat + p_pol) /
  (p_tradeoff + p_mc) /
  (p_sens + p_be) +
  plot_annotation(
    title = "Credit Limit Increase Optimization: Strategy & Validation Dashboard",
    theme = theme(plot.title = element_text(face = "bold", size = 14))
  )

ggsave("figures/r_dashboard.png", p_dash, width = 14, height = 14, dpi = 150)


## 7. Strategy Summary & Operationalization

Comparing the capital-constrained optimal strategy against the naive `offer_all` and heuristic `prime_near` baselines in the `base` macro scenario:

### Operational Recommendations
1. **Enforce Cooldown Rules**: Block increases when `Days Since Last Loan < 60` to eliminate historical policy violations.
2. **Deploy Segmented Loan Cutoffs**: Use full approvals for Prime, cap Near-Prime at $4,000, and cap Sub-Prime at $2,500.
3. **Dynamic Capital Management**: Re-estimate the shadow price of capital quarterly as macroeconomic indicators shift.
4. **Outcome Tracking**: Maintain real-time tracking of post-offer default, take-up, and repayment rates to continuously refine risk tier boundaries.


In [40]:
head_tbl <- results |>
  filter(strategy %in% c("offer_all", "prime_near", "dp_constrained")) |>
  select(strategy, npv, gross_profit, credit_loss, exposure_vs_cap, within_cap, default_rate_per_accepted) |>
  mutate(across(c(npv, gross_profit, credit_loss), round))

kable(head_tbl, digits = 3, format.args = list(big.mark = ","))




|strategy       |       npv| gross_profit| credit_loss| exposure_vs_cap|within_cap | default_rate_per_accepted|
|:--------------|---------:|------------:|-----------:|---------------:|:----------|-------------------------:|
|prime_near     | 1,059,317|    1,738,024|     678,707|           1.290|FALSE      |                     0.043|
|offer_all      | 1,066,904|    2,400,604|   1,333,700|           1.784|FALSE      |                     0.061|
|dp_constrained | 1,114,736|    1,645,996|     531,260|           1.000|FALSE      |                     0.050|